# Step 5: Clean and impute prescriptions

In [ ]:
import pyspark
import re
import dxpy
from scipy import stats
import hail as hl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import sys
import importlib
sys.path.append('../')
import prescriptions_processing

In [ ]:
importlib.reload(prescriptions_processing)

from prescriptions_processing import DataCleaning

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'filtered_prescriptions_with_doses_v6.2.0.ht'

first_output_database = 'prescriptions_db'
first_output_tb = 'cleaned_prescriptions_with_doses_v6.2.0.ht'

project_id = dxpy.PROJECT_CONTEXT_ID

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=project_id)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

In [ ]:
cleaner = DataCleaning(
    ht=ht,
    dose_std_dev_threshold=8,
    quantity_std_dev_threshold=10000, 
    quantity_threshold=1000,          
    impute_values=True                
)

In [ ]:
cleaned_ht = cleaner.clean_data()

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {first_output_database} LOCATION 'dnax://'")
first_output_db_id = dxpy.find_one_data_object(name=first_output_database, classname='database', project=project_id)['id']
first_output_url = f'dnax://{first_output_db_id}/{first_output_tb}'

%time cleaned_ht.write(first_output_url, overwrite=True)